In [1]:
import pandas as pd


In [2]:
ingredients = pd.read_csv('Ingredients_Utf8.csv')


print(ingredients.head())

   ingredient_id category subcategory     ingredient_name  bottle_size_ml  \
0              1   Liqour     Tequila    Jose Cuervo Gold          1000.0   
1              2   Liqour     Tequila    Jose Cuervo Gold           750.0   
2              3   Liqour     Tequila  Jose Cuervo Silver          1000.0   
3              4   Liqour     Tequila  Jose Cuervo Silver           750.0   
4              5   Liqour     Tequila         Olmeca Gold           750.0   

   bottle_cost  cost_per_ml  
0       4000.0     4.000000  
1       3500.0     4.666667  
2       4000.0     4.000000  
3       3500.0     4.666667  
4       4000.0     5.333333  


In [3]:
import pandas as pd
recipes = pd.read_csv('recipes.csv')
print(recipes.head())


  Cocktail_id cocktail_name  ingredient_id   ingredient_name  quantity
0       v01sb    Sea Breeze             56      Sminorff Red      60.0
1       v01sb    Sea Breeze            202  Grapefruit juice      30.0
2       v01sb    Sea Breeze            203   Cranberry juice      60.0
3       v01sb    Sea Breeze            215       Sugar syrup      15.0
4       v01sb    Sea Breeze            229         Ice-cubed     100.0


In [4]:
ingredientcost= recipes.quantity * ingredients.cost_per_ml
print(ingredientcost.head())


0    240.000000
1    140.000000
2    240.000000
3     70.000000
4    533.333333
dtype: float64


In [5]:
print(recipes.dtypes)
print(ingredients.dtypes)

Cocktail_id            str
cocktail_name          str
ingredient_id        int64
ingredient_name        str
quantity           float64
dtype: object
ingredient_id        int64
category               str
subcategory            str
ingredient_name        str
bottle_size_ml     float64
bottle_cost        float64
cost_per_ml        float64
dtype: object


In [6]:
recipes["ingredient_id"] = recipes["ingredient_id"].astype(str)
ingredients["ingredient_id"] = ingredients["ingredient_id"].astype(str)

In [7]:
cocktails = recipes.merge(
    ingredients,
    on="ingredient_id",
    how="left"
)

In [8]:
print(cocktails.head())

  Cocktail_id cocktail_name ingredient_id ingredient_name_x  quantity  \
0       v01sb    Sea Breeze            56      Sminorff Red      60.0   
1       v01sb    Sea Breeze           202  Grapefruit juice      30.0   
2       v01sb    Sea Breeze           203   Cranberry juice      60.0   
3       v01sb    Sea Breeze           215       Sugar syrup      15.0   
4       v01sb    Sea Breeze           229         Ice-cubed     100.0   

  category subcategory ingredient_name_y  bottle_size_ml  bottle_cost  \
0   Liqour       Vodka     Sminorff Red           1000.0       2500.0   
1    Mixer       Juice  Grapefruit juice          1000.0        400.0   
2    Mixer       Juice   Cranberry juice          1000.0        300.0   
3    Mixer       Syrup       Sugar syrup          1000.0        300.0   
4      Ice         Ice         Ice cubed          1000.0         50.0   

   cost_per_ml  
0         2.50  
1         0.40  
2         0.30  
3         0.30  
4         0.05  


In [9]:
cocktails["ingredient_cost"] = (
    cocktails["quantity"] *
    cocktails["cost_per_ml"]
)

In [10]:
cocktail_costs = (
    cocktails.groupby(
        ["Cocktail_id", "cocktail_name"]
    )["ingredient_cost"]
    .sum()
    .reset_index()
)

In [11]:
print(cocktails.columns.tolist())

['Cocktail_id', 'cocktail_name', 'ingredient_id', 'ingredient_name_x', 'quantity', 'category', 'subcategory', 'ingredient_name_y', 'bottle_size_ml', 'bottle_cost', 'cost_per_ml', 'ingredient_cost']


In [12]:
cocktail_costs.rename(
    columns={"ingredient_cost": "total_cost"},
    inplace=True
)

In [13]:
print(cocktail_costs.head())

  Cocktail_id     cocktail_name   total_cost
0       b01sc          Side car   231.250000
1       b02ba  Brandy Alexander   239.100000
2       b03sa           Sangria  6303.516667
3       b04bb               B&B   242.000000
4       b05bs    Brandy stinger   219.000000


In [14]:
cocktail_costs.to_csv(
    "cocktail_prices.csv",
    index=False
)

print(cocktail_costs.head())

  Cocktail_id     cocktail_name   total_cost
0       b01sc          Side car   231.250000
1       b02ba  Brandy Alexander   239.100000
2       b03sa           Sangria  6303.516667
3       b04bb               B&B   242.000000
4       b05bs    Brandy stinger   219.000000


In [15]:
def calculate_selling_price(total_cost):
    return total_cost * 4

In [16]:
total_cost = cocktail_costs["total_cost"]
spillage_pct = 0.05      # 5%
target_gp_pct = 0.75     # 75% GP
vat_pct = 0.16     # 16% VAT

# Adjust cost for wastage/spillage
adjusted_cost = total_cost * (1 + spillage_pct)

# Calculate selling price before VAT
selling_price_ex_vat = adjusted_cost / (1 - target_gp_pct)

# Add VAT
final_price = selling_price_ex_vat * (1 + vat_pct)


In [17]:
print (cocktail_costs.cocktail_name, round(final_price, 2))


0                   Side car
1           Brandy Alexander
2                    Sangria
3                        B&B
4             Brandy stinger
5          French connection
6             Cherry Blossom
7                 Black jack
8              Brandy cooler
9         Between the sheets
10           Classic Martini
11                  Gin Fizz
12                   Bramble
13           Singapore Sling
14                 Pink lady
15                     Bronx
16               French Kiss
17                White lady
18            Honolulu punch
19                   Negroni
20             Pimms cup no1
21                Caipirinha
22                Cuba libre
23          Classic_Daiquiri
24                   Mai_tai
25                pinacolada
26            Planters punch
27                   Barcadi
28                    Mojito
29                    Zombie
30      Long Island Iced Tea
31                 Hurricane
32                    Rumble
33                 Margarita
34           T

In [18]:
import pandas as pd

# Load  existing cocktail cost file
df = pd.read_csv("cocktail_prices.csv")

# Parameters
SPILLAGE_PCT = 0.05   # 5%
TARGET_GP_PCT = 0.65  # 65%
VAT_PCT = 0.16        # 16%

# Calculations
df["cost_after_spillage"] = df["total_cost"] * (1 + SPILLAGE_PCT)

df["selling_price_before_vat"] = (
    df["cost_after_spillage"] / (1 - TARGET_GP_PCT)
)

df["selling_price_after_vat"] = (
    df["selling_price_before_vat"] * (1 + VAT_PCT)
)

# Round to 2 decimal places
cols_to_round = [
    "total_cost",
    "cost_after_spillage",
    "selling_price_before_vat",
    "selling_price_after_vat"
]

df[cols_to_round] = df[cols_to_round].round(2)

# Save new CSV
df.to_csv("cocktail_final_prices.csv", index=False)

print(df.head())

  Cocktail_id     cocktail_name  total_cost  cost_after_spillage  \
0       b01sc          Side car      231.25               242.81   
1       b02ba  Brandy Alexander      239.10               251.05   
2       b03sa           Sangria     6303.52              6618.69   
3       b04bb               B&B      242.00               254.10   
4       b05bs    Brandy stinger      219.00               229.95   

   selling_price_before_vat  selling_price_after_vat  
0                    693.75                   804.75  
1                    717.30                   832.07  
2                  18910.55                 21936.24  
3                    726.00                   842.16  
4                    657.00                   762.12  


In [19]:
pip install streamlit

   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.2 MB ? eta -:--:--
   - -----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
